In [1]:
!pip install -q opencv-python matplotlib numpy pillow


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install -q accelerate


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [3]:
!python -m pip install --upgrade --force-reinstall git+https://github.com/huggingface/transformers

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-n5qioosp
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-n5qioosp
  Resolved https://github.com/huggingface/transformers to commit bdee0889714e9cb3e53d3b1b2a626919479d356c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached huggingface_hub-1.1.5-py3-none-any.whl.metadata (13 kB)
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pyyaml-6.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached regex-2025.11.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.met

In [1]:
%%bash
git lfs install
git clone https://huggingface.co/datasets/xiang709/VRSBench

Process is terminated.


In [2]:
import zipfile
for z in [ "VRSBench/Annotations_val.zip","VRSBench/Images_val.zip"]:
    with zipfile.ZipFile(z, 'r') as zip_ref:
        zip_ref.extractall("VRSBench_val")

In [1]:
LOCAL_ANNOTATION_DIR = "VRSBench_val/Annotations_val"
LOCAL_IMAGE_DIR = "VRSBench_val/Images_val"

In [1]:
import os
from huggingface_hub import hf_hub_download, login
import sys
import torch
login()

In [3]:
if not os.path.exists("GroundingDINO"):
    !git clone https://github.com/IDEA-Research/GroundingDINO.git
    %cd GroundingDINO
    !pip install -q -e .
    %cd ..

In [4]:
os.makedirs("weights", exist_ok=True)

if not os.path.exists("weights/groundingdino_swint_ogc.pth"):
    print("Downloading Grounding DINO weights...")
    !wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth -P weights/
else:
    print("Grounding DINO weights already exist.")

print("Weights Downloaded")
!ls -lh weights/

Grounding DINO weights already exist.
Weights Downloaded
total 662M
-rw-r--r-- 1 root root 662M Mar 21  2023 groundingdino_swint_ogc.pth


In [5]:
%cd /home/GroundingDINO
!pip install -e .
%cd /content
print("Grounding DINO re-compiled for GPU.")

/root/miniconda3/envs/py3.10/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/home/GroundingDINO
Obtaining file:///home/GroundingDINO
  Preparing metadata (setup.py) ... done
  DEPRECATION: Legacy editable install of groundingdino==0.1.0 from file:///home/GroundingDINO (setup.py develop) is deprecated. pip 25.3 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457
  Running setup.py develop for groundingdino
    error: subprocess-exited-with-error
    
    × python setup.py develop did not run successfully.
    │ exit code: 1
    ╰─> [117 lines of output]
        Compiling with CUDA
        running develop
        /root/miniconda3/envs/py3.10/lib/python3.10/site-packages/setuptools/command/develop.py:41: EasyInstallDeprecationWarning: easy_in

/root/miniconda3/envs/py3.10/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


In [12]:
import cv2
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.ops import box_convert
import json
import glob
from transformers.modeling_utils import ModuleUtilsMixin

try:
    from transformers import (
        AutoProcessor, 
        AutoModelForZeroShotObjectDetection,
        Sam3Model, 
        Sam3Processor
    )
    print("Loaded Transformers Classes")
except ImportError as e:
    raise ImportError(f"Transformers import failed: {e}. Please run the installation cell again.")


if not hasattr(ModuleUtilsMixin, 'get_head_mask'):
    print("Patching 'get_head_mask' for Grounding DINO compatibility...")
    
    def get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
        if head_mask is not None:
            head_mask = self._convert_head_mask_to_5d(head_mask, num_hidden_layers)
            if is_attention_chunked is True:
                head_mask = head_mask.unsqueeze(-1)
        else:
            head_mask = [None] * num_hidden_layers
        return head_mask

    ModuleUtilsMixin.get_head_mask = get_head_mask
    print("Patch applied. DINO should work now.")


Loaded Transformers Classes


In [13]:
from torch.utils.data import Dataset
from PIL import Image

ANNOTATION_DIR = "VRSBench_val/Annotations_val"
IMAGE_DIR = "VRSBench_val/Images_val"
OUTPUT_DIR = "eval_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

class VRSBenchDataset:
    def __init__(self, anno_dir, img_dir, max_samples=5):
        self.files = glob.glob(os.path.join(anno_dir, "*.json"))[:max_samples]
        self.img_dir = img_dir
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        with open(self.files[idx], 'r') as f: data = json.load(f)
        img_name = data.get('image') or os.path.basename(self.files[idx]).replace('.json','.png')
        
        # Fix extensions logic
        img_path = os.path.join(self.img_dir, img_name)
        if not os.path.exists(img_path): 
            alt_path = img_path.replace('.png','.jpg')
            if os.path.exists(alt_path): img_path = alt_path
        
        objs = data.get('objects', [])
        return {"image_path": img_path, "image_id": img_name, "label": objs[0]['obj_cls'] if objs else None}

dataset = VRSBenchDataset(ANNOTATION_DIR, IMAGE_DIR, 5)
  
  

--- Initializing Pipeline on cuda ---
Loading DINO: IDEA-Research/grounding-dino-tiny...


Loading weights:   0%|          | 0/990 [00:00<?, ?it/s]

Loading SAM: facebook/sam3...


Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



   -> Found 3 boxes, Extracted 28 OBBs
   -> Saved visualization: eval_results/pred_P1410_0020.png
[3] Processing 11243_0000.png (Target: dam)
   -> Found 1 boxes, Extracted 1 OBBs
   -> Saved visualization: eval_results/pred_11243_0000.png
[4] Processing 06759_0000.png (Target: golffield)
   (No objects found for 'golffield')
[5] Processing P2337_0016.png (Target: storage-tank)
   -> Found 2 boxes, Extracted 33 OBBs
   -> Saved visualization: eval_results/pred_P2337_0016.png


In [25]:
class RefinedPipeline:
    def __init__(self, device="cuda"):
        self.device = device
        print(f"--- Initializing Refined Pipeline ---")
        
        # DINO (Detector)
        dino_id = "IDEA-Research/grounding-dino-tiny"
        self.dino_processor = AutoProcessor.from_pretrained(dino_id)
        self.dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(dino_id).to(device)
        
        # SAM 3 (Refiner)
        sam_id = "facebook/sam3" 
        self.sam_processor = Sam3Processor.from_pretrained(sam_id)
        self.sam_model = Sam3Model.from_pretrained(sam_id).to(device)
        print("Models Loaded")

    def get_obb_from_mask(self, mask_bool):
        mask_uint8 = mask_bool.astype(np.uint8) * 255
        contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours: return None
        largest_cnt = max(contours, key=cv2.contourArea)
        
        # Filter noise (ignore tiny dots)
        if cv2.contourArea(largest_cnt) < 50: return None
        
        return cv2.minAreaRect(largest_cnt)

    def run(self, image_path, text_prompt, output_file, box_thresh=0.25):
        # 1. Load Image
        try:
            image_pil = Image.open(image_path).convert("RGB")
        except: return
        image_cv2 = np.array(image_pil)
        h, w, _ = image_cv2.shape

        # 2. Grounding DINO (Find Coarse Boxes)
        prompt = text_prompt.lower().strip()
        if not prompt.endswith("."): prompt += "."
        
        dino_inputs = self.dino_processor(images=image_pil, text=prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            dino_outputs = self.dino_model(**dino_inputs)

        results = self.dino_processor.image_processor.post_process_object_detection(
            dino_outputs, threshold=box_thresh, target_sizes=[(h, w)]
        )[0]
        boxes = results["boxes"] # [N, 4]

        if len(boxes) == 0:
            print(f"   (No objects found)")
            return

        # 3. SAM 3 (Refine Boxes -> Best Mask)
        input_boxes = [boxes.tolist()] 
        sam_inputs = self.sam_processor(images=image_pil, input_boxes=input_boxes, return_tensors="pt").to(self.device)
        
        with torch.no_grad():
            sam_outputs = self.sam_model(**sam_inputs)

        all_scores = None
        possible_keys = ["iou_scores", "iou_predictions", "scores", "pred_iou"]
        
        # Check attributes
        for key in possible_keys:
            if hasattr(sam_outputs, key):
                all_scores = getattr(sam_outputs, key)
                # print(f"   DEBUG: Found scores in '{key}'")
                break
        
        # Fallback if scores are missing
        if all_scores is None:
            print(f"Warning: Could not find scores. Output keys: {sam_outputs.keys() if hasattr(sam_outputs, 'keys') else 'Unknown'}")
            # Default to picking the first mask (index 0) for every box
            device = sam_outputs.pred_masks.device
            n_boxes = sam_outputs.pred_masks.shape[1]
            all_scores = torch.zeros((1, n_boxes, 3), device=device)
            all_scores[:, :, 0] = 1.0 

        # 4. Process Masks & OBBs
        all_masks = sam_outputs.pred_masks[0]        # [N, 3, H, W]
        batch_scores = all_scores[0]                 # [N, 3]
        
        # Resize masks to image size
        all_masks_resized = torch.nn.functional.interpolate(
            all_masks, size=(h, w), mode="bilinear", align_corners=False
        )

        # Selection Loop
        count = 0
        for i in range(len(boxes)):
            # A. Pick the BEST mask index for this specific box
            scores = all_scores[i] # [3]
            best_idx = torch.argmax(scores).item()
            
            # B. Get that specific mask
            best_mask_logits = all_masks_resized[i, best_idx, :, :] # [H, W]
            binary_mask = (best_mask_logits > 0.0).cpu().numpy()
            
            # C. Get OBB
            obb = self.get_obb_from_mask(binary_mask)
            
            if obb:
                count += 1
                # Draw DINO Box (Blue, Thin) - Optional, just to show the "Before"
                # x1, y1, x2, y2 = map(int, boxes[i].tolist())
                # cv2.rectangle(image_cv2, (x1, y1), (x2, y2), (255, 0, 0), 1)
                
                # Draw SAM OBB (Red, Thick) - The "After"
                box_points = cv2.boxPoints(obb)
                box_points = np.int0(box_points)
                cv2.drawContours(image_cv2, [box_points], 0, (255, 0, 0), 2) # Red

        print(f"   -> Processed {len(boxes)} DINO boxes -> Generated {count} Refined OBBs")
        Image.fromarray(image_cv2).save(output_file)

# --- EXECUTION ---
ANNOTATION_DIR = "VRSBench_val/Annotations_val"
IMAGE_DIR = "VRSBench_val/Images_val"
OUTPUT_DIR = "eval_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

pipeline = RefinedPipeline()

# Get Dataset
json_files = glob.glob(os.path.join(ANNOTATION_DIR, "*.json"))[:5]


--- Initializing Refined Pipeline ---


Loading weights:   0%|          | 0/990 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

